In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, time
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt

INPUT,WORK=Path('/kaggle/input'),Path('/kaggle/working')
def newest(pattern,label):
    paths=list(INPUT.rglob(pattern))
    if not paths: raise FileNotFoundError(f'Attach saved output containing {label}: {pattern}')
    return max(paths,key=lambda p:p.stat().st_mtime)

binary_sources=[]
for mp in INPUT.rglob('simulation_metadata.json'):
    if mp.parent.name!='binary': continue
    try: meta=json.loads(mp.read_text(encoding='utf-8'))
    except Exception: continue
    if meta.get('family')=='binary' and int(meta.get('parameter_draws_unique',0))>=100_000:
        binary_sources.append((meta.get('created_utc',''),mp.parent))
if not binary_sources: raise FileNotFoundError('Attach Day 7 Cosmic swan Bayesian binary simulations.')
_,BINARY_DIR=sorted(binary_sources,key=lambda x:x[0])[-1]
observations=np.load(BINARY_DIR/'observations.npy',mmap_mode='r')
parameters=pd.read_csv(BINARY_DIR/'parameters.csv')
time_days=np.asarray(np.load(BINARY_DIR/'time_days.npy'),dtype=float)
if observations.ndim!=2 or observations.shape!=(len(parameters),len(time_days)):
    raise ValueError('Day 7 observations, parameters, and time axis do not match.')
MAP_PATH=newest('day9_12_conditional_mle_parameters.csv','Day 9.12 conditional MLE estimates')
DIAG_PATH=newest('day9_12_conditional_mle_diagnostics.csv','Day 9.12 conditional MLE diagnostics')
map_df=pd.read_csv(MAP_PATH)
case_df=pd.read_csv(DIAG_PATH)
TARGETS=[(87483,'duration_fraction'),(4923,'noise_sigma')]
SEEDS={}
for row,name in TARGETS:
    sub=map_df[map_df.simulation_row.astype(int).eq(row)]
    vals=sub.set_index('parameter').estimate.astype(float).to_dict()
    if 'phase' not in vals or name not in vals:
        raise ValueError(f'Missing Day 9.12 values for row {row}, profile parameter {name}.')
    match=case_df[case_df.simulation_row.astype(int).eq(row)]
    if len(match)!=1 or not bool(match.iloc[0].optimizer_success):
        raise ValueError(f'Day 9.12 conditional fit for row {row} is missing or did not succeed.')
    vals['period_fixed_days']=float(match.iloc[0].period_fixed_days)
    if row>=len(parameters): raise ValueError(f'Simulation row {row} is out of range.')
    SEEDS[row]=vals
OUT=WORK/('day9-15-boundary-profile-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True,exist_ok=False)
print('[PASS] Day 7:',BINARY_DIR)
print('[PASS] Day 9.12 estimates:',MAP_PATH.parent)
print('Profile targets:',TARGETS)
print('Fixed periods:',{r:SEEDS[r]['period_fixed_days'] for r,_ in TARGETS})
print('Output:',OUT)

def smooth_clip_np(x,sharpness=500.0):
    return (np.logaddexp(0.0,sharpness*x)-np.logaddexp(0.0,sharpness*(x-1.0)))/sharpness

def predict_np(t,baseline,period,phase,depth,ratio,duration,ellip,reflection):
    orb=np.mod(t/period-phase+0.5,1.0)-0.5
    sec=np.mod(orb,1.0)-0.5
    half=max(float(duration)/2.0,1e-6)
    ingress=max(half*0.18,1e-6)
    primary=smooth_clip_np((half+ingress-np.abs(orb))/(2.0*ingress))
    secondary=smooth_clip_np((half+ingress-np.abs(sec))/(2.0*ingress))
    return baseline-depth*primary-depth*ratio*secondary+ellip*np.cos(4*np.pi*orb)+reflection*np.cos(2*np.pi*orb)

In [ ]:
profile_rows=[]
plot_data=[]
for row,profile_parameter in TARGETS:
    started=time.time()
    seed=SEEDS[row]
    period=float(seed['period_fixed_days'])
    t=np.asarray(time_days,dtype=float)
    y=np.asarray(observations[row],dtype=float)
    good=np.isfinite(t)&np.isfinite(y)
    t,y=t[good],y[good]
    order=np.argsort(t)
    t,y=t[order],y[order]
    t=t-t.min()
    theta_seed=np.array([seed['baseline_flux'],seed['phase'],seed['depth_primary'],seed['depth_ratio'],
        seed['duration_fraction'],seed['ellipsoidal_amplitude'],seed['reflection_amplitude'],
        np.log(seed['noise_sigma'])],dtype=float)
    bounds=[(0.95,1.05),(0.0,1.0),(0.002,0.45),(0.0,1.0),(0.005,0.12),
            (0.0,0.02),(-0.01,0.01),(np.log(1e-4),np.log(0.03))]
    truth=parameters.iloc[row]
    if profile_parameter=='duration_fraction':
        truth_profile=float(truth['duration_fraction'])
        
        grid=np.unique(np.r_[np.linspace(0.06,0.115,28),np.linspace(0.115,0.12,121)])
        fixed_index=4
        x_label='Duration fraction'
        truth_label='known truth (evaluation only)'
    else:
        truth_profile=float(truth['noise_sigma'])
        center=float(seed['noise_sigma'])
        lo=max(1.0001e-4,center/2.0)
        hi=min(0.02999,center*2.0)
        grid=np.unique(np.r_[np.geomspace(1.0001e-4,0.02999,45),np.geomspace(lo,hi,81)])
        fixed_index=7
        x_label='Noise sigma'
        truth_label='known truth (evaluation only)'
    values=[]
    for fixed_value in grid:
        theta0=theta_seed.copy()
        theta0[fixed_index]=fixed_value if fixed_index==4 else np.log(fixed_value)
        free_indices=[i for i in range(8) if i!=fixed_index]
        free_bounds=[bounds[i] for i in free_indices]
        starts=[]
        starts.append(theta0.copy())
        shifted=theta0.copy(); shifted[1]=(shifted[1]+0.5)%1.0; starts.append(shifted)
        def objective(free):
            theta=theta0.copy()
            theta[free_indices]=free
            pred=predict_np(t,theta[0],period,theta[1],theta[2],theta[3],theta[4],theta[5],theta[6])
            z=(y-pred)/np.exp(theta[7])
            return float(0.5*np.sum(z*z+2.0*theta[7]+np.log(2.0*np.pi)))
        fits=[minimize(objective,x[free_indices],method='L-BFGS-B',bounds=free_bounds,
            options={'maxiter':1200,'ftol':1e-10}) for x in starts]
        finite=[fit for fit in fits if np.isfinite(fit.fun)]
        if not finite: raise RuntimeError(f'No finite profile fit for row {row} at {fixed_value}.')
        best=min(finite,key=lambda fit:fit.fun)
        fixed_output=float(fixed_value)
        values.append({'simulation_row':row,'profile_parameter':profile_parameter,
            'fixed_value':fixed_output,'profile_nll':float(best.fun),
            'optimizer_success':bool(best.success),'optimizer_message':str(best.message),
            'successful_starts':sum(bool(f.success) and np.isfinite(f.fun) for f in fits)})
    minimum=min(item['profile_nll'] for item in values)
    for item in values:
        item['delta_profile_nll']=float(item['profile_nll']-minimum)
        profile_rows.append(item)
    best=min(values,key=lambda item:item['profile_nll'])
    grid_tol=(float(grid[-1])-float(grid[0]))/(len(grid)-1)*0.51
    at_edge=abs(best['fixed_value']-float(grid[0]))<=grid_tol or abs(best['fixed_value']-float(grid[-1]))<=grid_tol
    delta_truth=float(np.interp(truth_profile,[v['fixed_value'] for v in values],[v['profile_nll'] for v in values])-minimum)
    # Keep a local profile-likelihood reference band; do not interpret it as a posterior interval.
    inside=[v['fixed_value'] for v in values if v['delta_profile_nll']<=1.92]
    summary={'simulation_row':row,'profile_parameter':profile_parameter,'fixed_period_days':period,
        'profile_minimum':best['fixed_value'],'known_truth_evaluation_only':truth_profile,
        'profile_delta_nll_at_truth_interpolated':delta_truth,'minimum_at_grid_edge':bool(at_edge),
        'grid_low':float(grid[0]),'grid_high':float(grid[-1]),'grid_points':int(len(grid)),
        'delta_nll_le_1_92_grid_low':float(min(inside)) if inside else float('nan'),
        'delta_nll_le_1_92_grid_high':float(max(inside)) if inside else float('nan'),
        'failed_optimizer_count':int(sum(not v['optimizer_success'] for v in values)),
        'seconds':round(time.time()-started,1)}
    plot_data.append((row,profile_parameter,np.array([v['fixed_value'] for v in values]),
        np.array([v['delta_profile_nll'] for v in values]),truth_profile,summary))
    print('Profile result:',summary,flush=True)

profile_df=pd.DataFrame(profile_rows)
summary_df=pd.DataFrame([item[-1] for item in plot_data])
profile_df.to_csv(OUT/'day9_15_nuisance_profile_grid.csv',index=False)
summary_df.to_csv(OUT/'day9_15_nuisance_profile_summary.csv',index=False)
fig,axes=plt.subplots(1,len(plot_data),figsize=(13,4.5),squeeze=False,constrained_layout=True)
for ax,(row,name,x,delta,truth_value,summary) in zip(axes.flat,plot_data):
    ax.plot(x,delta,color='navy',marker='.',ms=3,lw=1)
    ax.axhline(1.92,color='gray',ls=':',lw=1,label='profile reference ΔNLL=1.92')
    ax.axvline(truth_value,color='crimson',ls='--',lw=1,label='known truth (evaluation only)')
    ax.axvline(summary['profile_minimum'],color='darkgreen',ls='-',lw=1,label='profile minimum')
    ax.set_title(f'Row {row} | profile {name}')
    ax.set_xlabel('Noise sigma' if name=='noise_sigma' else 'Duration fraction')
    ax.set_ylabel('Δ profile NLL')
    if name=='noise_sigma': ax.set_xscale('log')
    ax.legend(fontsize=8)
fig.savefig(OUT/'day9_15_nuisance_profiles.png',dpi=150)
plt.show()
report={'created_utc':datetime.now(timezone.utc).isoformat(),'day9_12_input':str(MAP_PATH.parent),
    'targets':summary_df.to_dict(orient='records'),
    'limits':['These are one-dimensional conditional profile likelihoods, not Bayesian posterior intervals.',
        'The profile fixes the period at the Day 9.10 data-selected minimum.',
        'Known truth is evaluation-only and is not used by the optimizer.',
        'Do not widen parameter support without checking the simulation design and scientific assumptions.'],
    'outputs':['day9_15_nuisance_profile_grid.csv','day9_15_nuisance_profile_summary.csv','day9_15_nuisance_profiles.png']}
(OUT/'day9_15_profile_summary.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print(summary_df.to_string(index=False))
print(json.dumps(report,indent=2))
print('Outputs:',OUT)